<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/4_model_training/4_4_model_catboost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4_4_model_catboost

In [3]:
%pip uninstall -y catboost -q

# Alineamos el stack numérico y CatBoost en una sola transacción
%pip install --no-cache-dir -U \
  numpy==2.1.2 \
  scipy==1.13.1 \
  scikit-learn==1.5.2 \
  catboost==1.2.8 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 326.9 MB/s eta 0:00:00


In [1]:
from catboost import CatBoostRegressor, Pool
print("CatBoost OK")

CatBoost OK


## Introducción y Resumen

El objetivo de esta notebook es generar las ventanas de entrada (X) y targets (y) para los horizontes de 30, 60 y 90 minutos, aplica el escalado de features sobre cada conjunto (train, valid, test) y finalmente guarda las ventanas escaladas en disco, dejándolas listas para el entrenamiento de modelos.

0. Configuración del Entorno

    Se conecta Google Drive y se clona el repositorio de trabajo. Se instalan e importan librerías necesarias como pandas, numpy, matplotlib y seaborn. Se cargan los datasets procesados previamente y se muestra un resumen de la información de los datasets.

1. Carga de datos

    Se importan los datasets procesados `mnq_train`, `mnq_valid` y `mnq_test`. Se revisa su estructura (filas, columnas, tipos de datos) y también de importa el listado de features seleccionados para cada ventana de tiempo.





## 0. Configuración del Entorno


### 0.1. Clonado de repositorio / Acceso a Drive

In [1]:
#Clonamos el repo
#LINK DE REPOSITORIO: https://github.com/GUNAPILLCO/neural_profit
#!git clone https://github.com/GUNAPILLCO/neural_profit.git

In [2]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 0.2. Instalación de librerías


In [3]:
import catboost
from catboost import CatBoostRegressor, Pool

### 0.3. Importación de librerías


In [4]:
# ==============================
# Librerías estándar de Python
# ==============================
import os
import sys
import re
import glob
import warnings
import requests
from datetime import datetime, timedelta
from functools import reduce

# ==============================
# Manejo y procesamiento de datos
# ==============================
import pandas as pd
import numpy as np
from tabulate import tabulate

# ==============================
# Visualización
# ==============================
import matplotlib.pyplot as plt

# ==============================
# Estadística
# ==============================
from scipy.stats import spearmanr

# ==============================
# Machine Learning y utilidades
# ==============================
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import sklearn, scipy
import sklearn, numpy, scipy #optuna
#from sklearn.ensemble import RandomForestRegressor

import joblib
#import optuna
from tqdm import tqdm

# ==============================
# Configuración general
# ==============================
warnings.filterwarnings("ignore")

import time

import xgboost as xgb
#from xgboost import XGBRegressor

import sys, platform, lightgbm as lgb
import numpy as np, pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
#from lightgbm import LGBMRegressor


In [5]:
print("python:", sys.version)
print("Platform:", platform.platform())
print("numpy:", numpy.__version__)
print("scipy:", scipy.__version__)
print("sklearn:", sklearn.__version__)
#print("optuna:", optuna.__version__)
print("xgboost:", xgb.__version__)
print("lightgbm:", lgb.__version__)

# CatBoost usa la clase para exponer versión
print("catboost:", catboost.__version__)

python: 3.12.11 (main, Jun  4 2025, 08:56:18) [GCC 11.4.0]
Platform: Linux-6.6.97+-x86_64-with-glibc2.35
numpy: 2.1.2
scipy: 1.13.1
sklearn: 1.5.2
xgboost: 3.0.5
lightgbm: 4.6.0
catboost: 1.2.8


## 1. Carga de datos

### 1.1. Carga de datasets `mnq_train`, `mnq_valid` y `mnq_test`






In [6]:
def load_data(data: str):

    data_path = f'{drive_path}/3_dataset_preparation/mnq_{data}.parquet'
    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(data_path)

    # Asegurar que el índice esté en formato datetime
    df.index = pd.to_datetime(df.index)

    # Crear una nueva columna 'date' con la fecha extraída del índice
    df['date'] = df.index.date

    # Reordenar columnas: 'date', 'time_str', y luego el resto
    cols = ['date'] + [col for col in df.columns if col not in ['date']]

    df = df[cols]

    return df

In [7]:
#mnq_train = load_data("train")
#mnq_valid = load_data("valid")
#mnq_test = load_data("test")

### 1.2. Información de datasets


In [8]:
def info_dataset(df, name: str):
  print(f"Información del dataset {name}:\n")

  # Contar valores únicos en la columna 'date'
  num_dias = df['date'].nunique()
  print(f"\tCantidad de días: {num_dias}")

  # Filtrar valores válidos
  validos_por_dia = df.dropna(subset=['close']).groupby('date').size()

  # Calcular el promedio
  promedio_por_fecha = validos_por_dia.mean()
  print(f"\tRegistros por día: {int(promedio_por_fecha)}")

  primer_hora = df.index[0].strftime('%H:%M')
  ultima_hora = df.index[-1].strftime('%H:%M')
  zona_horaria = df.index[0].tzinfo


  print(f"\tHora diaria de inicio {primer_hora}")
  print(f"\tHora diaria de final {ultima_hora}")
  print(f"\tZona horaria: {zona_horaria}\n")

  return num_dias, promedio_por_fecha

In [9]:
#info_dataset(mnq_train, 'mnq_train')
#info_dataset(mnq_valid, 'mnq_valid')
#info_dataset(mnq_test, 'mnq_test')

### 1.3. Carga de listado de features por ventana de tiempo

In [10]:
import json

# Ruta al archivo guardado
path = f'{drive_path}/2_feature_engineering/features_list.json'

with open(path, "r") as f:
    features_dict = json.load(f)

# Extraer las listas
#features_to_30 = features_dict["features_to_30"]
#features_to_60 = features_dict["features_to_60"]
#features_to_90 = features_dict["features_to_90"]


In [11]:
#print(f'Listado de features para 30min: {features_to_30}')
#print(f'Listado de features para 60min: {features_to_60}')
#print(f'Listado de features para 90min: {features_to_90}')

## 2. Carga de ventanas `X_train_*_scaled`, `X_valid_*_scaled`, `X_test_*_scaled`

### 2.0. Funciones

#### Función para cargar ventanas

In [12]:
def load_windows_and_scaler(target: str, scaled=True):
    """
    Carga datasets (X, y) para train, valid y test junto con el scaler global.

    Parámetros
    ----------
    drive_path : str
        Ruta base donde se encuentran los archivos.
    scaled : bool, default=True
        Si True busca en la carpeta 'ventanas_x_y_scaled',
        si False en 'ventanas_x_y'.

    Retorna
    -------
    X_train, y_train, X_valid, y_valid, X_test, y_test, scaler
    """

    #Ruta de ventandas escaladas
    path_train  = f'{drive_path}/3_dataset_preparation/xy_windows_scaled/xy_train_{target}_scaled.npz'
    path_valid  = f'{drive_path}/3_dataset_preparation/xy_windows_scaled/xy_valid_{target}_scaled.npz'
    path_test   = f'{drive_path}/3_dataset_preparation/xy_windows_scaled/xy_test_{target}_scaled.npz'

    #Ruta de escalador
    path_scaler = f"{drive_path}/3_dataset_preparation/global_scaler_{target}.pkl"

    # Cargar npz
    data_train = np.load(path_train)
    data_valid = np.load(path_valid)
    data_test  = np.load(path_test)

    # Extraer X, y
    X_train, y_train = data_train["X"], data_train["y"]
    X_valid, y_valid = data_valid["X"], data_valid["y"]
    X_test,  y_test  = data_test["X"],  data_test["y"]

    # Cargar scaler
    scaler = joblib.load(path_scaler)

    return X_train, y_train, X_valid, y_valid, X_test, y_test, scaler


#### Función para revisar información de ventanas

In [13]:
def xy_info(target: str, X_train, y_train, X_valid, y_valid, X_test, y_test):
    print(f'Información para horizonte de {target} minutos:')

    for name, X, y in [
        ("entrenamiento", X_train, y_train),
        ("validación", X_valid, y_valid),
        ("testeo", X_test, y_test),
    ]:
        print(f'\nSet de {name}:')
        print(f'\t{X.shape[0]} ventanas (n_samples).')

        if X.ndim == 2:

            print(f'\t{X.shape[1]} features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_{target})')
        elif X.ndim == 3:

            print(f'\t{X.shape[1]} pasos en lookback × {X.shape[2]} features por paso. Dimensión 3D: (n_samples, window_size, len(features_{target})')

        print(f'\t{y.shape[0]} targets.')
        print(f'\tDistribución y: mean={y.mean():.6f}, std={y.std():.6f}, min={y.min():.6f}, max={y.max():.6f}')

    return  X_train.shape[0], X_valid.shape[0], X_test.shape[0]

### 2.1 Carga de ventanas 30 minutos

In [14]:
X_train_30_scaled, y_train_30, X_valid_30_scaled, y_valid_30, X_test_30_scaled, y_test_30, scaler_30 = load_windows_and_scaler(target = '30')

In [15]:
n_samples_train_30, n_samples_valid_30, n_samples_test_30 = xy_info( '30', X_train_30_scaled, y_train_30, X_valid_30_scaled, y_valid_30, X_test_30_scaled, y_test_30)

Información para horizonte de 30 minutos:

Set de entrenamiento:
	193487 ventanas (n_samples).
	900 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_30)
	193487 targets.
	Distribución y: mean=0.000056, std=0.002760, min=-0.028931, max=0.032372

Set de validación:
	41567 ventanas (n_samples).
	900 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_30)
	41567 targets.
	Distribución y: mean=0.000103, std=0.003273, min=-0.023167, max=0.068056

Set de testeo:
	41567 ventanas (n_samples).
	900 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_30)
	41567 targets.
	Distribución y: mean=-0.000023, std=0.002407, min=-0.015982, max=0.016097


### 2.2 Carga de ventanas 60 minutos

In [16]:
X_train_60_scaled, y_train_60, X_valid_60_scaled, y_valid_60, X_test_60_scaled, y_test_60, scaler_60 = load_windows_and_scaler(target = '60')

In [17]:
n_samples_train_60, n_samples_valid_60, n_samples_test_60 = xy_info( '60', X_train_60_scaled, y_train_60, X_valid_60_scaled, y_valid_60, X_test_60_scaled, y_test_60)

Información para horizonte de 60 minutos:

Set de entrenamiento:
	193487 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_60)
	193487 targets.
	Distribución y: mean=0.000114, std=0.003907, min=-0.040179, max=0.036224

Set de validación:
	41567 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_60)
	41567 targets.
	Distribución y: mean=0.000163, std=0.004643, min=-0.038084, max=0.079896

Set de testeo:
	41567 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_60)
	41567 targets.
	Distribución y: mean=-0.000068, std=0.003521, min=-0.018494, max=0.020365


### 2.3 Carga de ventanas 90 minutos

In [18]:
X_train_90_scaled, y_train_90, X_valid_90_scaled, y_valid_90, X_test_90_scaled, y_test_90, scaler_90 = load_windows_and_scaler(target = '90')

In [19]:
n_samples_train_90, n_samples_valid_90, n_samples_test_90 = xy_info('90', X_train_90_scaled, y_train_90, X_valid_90_scaled, y_valid_90, X_test_90_scaled, y_test_90)

Información para horizonte de 90 minutos:

Set de entrenamiento:
	193487 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_90)
	193487 targets.
	Distribución y: mean=0.000159, std=0.004746, min=-0.037742, max=0.039284

Set de validación:
	41567 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_90)
	41567 targets.
	Distribución y: mean=0.000224, std=0.005887, min=-0.040748, max=0.083184

Set de testeo:
	41567 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_90)
	41567 targets.
	Distribución y: mean=-0.000132, std=0.004558, min=-0.023392, max=0.026213


## 3. Dataset de Métricas

Dado que cada entrenamiento demanda un tiempo considerable, antes de proceder verificaremos si ya existe un resultado previo de este modelo consultando el dataset de métricas.

### 3.1. Función para cargar métricas o generar dataset

In [20]:
def load_metrics(data: str):
    data_path = f'{drive_path}/4_model_training/{data}.parquet'
    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(data_path)
    return df

In [21]:
def metrics_verify(data: str) -> bool:
    data_path = f"{drive_path}/4_model_training/{data}.parquet"
    return os.path.exists(data_path)


In [22]:
def load_or_create_metrics (data:str):
  if metrics_verify(data):
      print(f"Las métricas existen y son almacenadas en {data[4:len(data)]}")
      model_metrics = load_metrics(data)
      #print(random_forest_metrics)
      metrics = True
  else:
      print(f"Las métricas no existen. Se crea el dataset {data[4:len(data)]} para almacenar las métricas")
      #Creamos la tabla para almacenar las métricas
      model_metrics = pd.DataFrame(columns=["RMSE", "MAE", "R2", "SMAPE", "DirAcc"])
      metrics = False

  return model_metrics, metrics

In [23]:
cat_metrics, metrics = load_or_create_metrics("4_4_catboost_metrics")

Las métricas no existen. Se crea el dataset catboost_metrics para almacenar las métricas


### 3.2. Función para guardar métricas

In [24]:
def save_metrics (metrics,  metrics_name: str):   #("4_2_xgboost_metrics")
  metrics_path = f"{drive_path}/4_model_training/{metrics_name}.parquet"
  metrics.to_parquet(metrics_path, index = True)
  print(f"Métricas guardadas en {metrics_path}")

### 3.3. Función para calcular las métricas

In [25]:
def evaluate_model(model, X, y_true, y_pred=None, eps=1e-8):
    """
    Evalúa RMSE, MAE, R2, SMAPE y DirAcc.
    - Si y_pred es None, predice con el modelo usando X.
    - Evita mean_squared_error(squared=...) para máxima compatibilidad.
    """
    if y_pred is None:
        y_pred = model.predict(X)

    # Asegurar 1D
    y_true = np.ravel(y_true)
    y_pred = np.ravel(y_pred)

    # RMSE sin sklearn
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2  = float(r2_score(y_true, y_pred))

    # SMAPE
    smape_val = 100.0 * np.mean(
        (np.abs(y_true - y_pred) / ((np.abs(y_true) + np.abs(y_pred)) / 2.0 + eps))
    )

    # Directional Accuracy
    directional_acc = float(np.mean(np.sign(y_true) == np.sign(y_pred)))

    return {
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "SMAPE": float(smape_val),
        "DirAcc": directional_acc
    }

In [26]:
def print_metrics(metrics, target:str):
  print(f"Métricas de {target}:\n")
  for k, v in metrics.items():
      print(f"\t{k:>5}:\t {float(v):.6f}")

## 4. Definición de modelo


### 4.1. Función de entrenamiento para modelo

In [27]:
def train_model_catboost(
    best_params,
    X_train, y_train,
    X_valid, y_valid,
    *,
    cat_features=None,
    use_gpu=False,
    iterations=5000,
    od_wait=200,
    verbose_every=200
):
    """
    Entrena un CatBoostRegressor con early stopping y devuelve (modelo, preds_valid).

    Parámetros:
      - best_params: dict con hiperparámetros propios de CatBoost (p.ej., depth, learning_rate, l2_leaf_reg, rsm, subsample, etc.)
      - X_train, y_train, X_valid, y_valid: datasets
      - cat_features: lista de índices o nombres de columnas categóricas (opcional)
      - use_gpu: bool, usa GPU si está disponible
      - iterations: número máximo de iteraciones (equivalente a n_estimators en LGBM)
      - od_wait: paciencia para early stopping (similar a early_stopping_rounds)
      - verbose_every: frecuencia de logs

    Comportamiento:
      - Usa 'use_best_model=True' para retener la mejor iteración.
      - predict() sobre el valid_pool ya utiliza la mejor iteración.
    """
    # Copiamos y completamos defaults sin tocar el dict original
    params = dict(best_params or {})
    params.setdefault("loss_function", "RMSE")
    params.setdefault("eval_metric", "RMSE")
    params.setdefault("random_seed", 42)
    params.setdefault("use_best_model", True)
    params.setdefault("od_type", "Iter")
    params.setdefault("od_wait", od_wait)
    params.setdefault("verbose", verbose_every)
    params.setdefault("thread_count", -1)  # usar todos los hilos

    # Iteraciones máximas (solo si no las seteaste ya en best_params)
    params.setdefault("iterations", iterations)

    # GPU opcional
    if use_gpu:
        params["task_type"] = "GPU"
        # params["devices"] = "0"  # descomentar si querés fijar dispositivo

    # Pools (eficientes y permiten cat_features)
    train_pool = Pool(X_train, y_train, cat_features=cat_features)
    valid_pool  = Pool(X_valid, y_valid, cat_features=cat_features)

    model = CatBoostRegressor(**params)
    model.fit(train_pool, eval_set=valid_pool)

    # Con use_best_model=True, ya queda en la mejor iteración
    preds = model.predict(valid_pool)

    return model, preds

### 4.2. Parámetros por defecto para modelo


In [28]:
catboost_default_params = {
    "loss_function": "RMSE",
    "eval_metric": "RMSE",
    "learning_rate": 0.02,
    "depth": 8,
    "l2_leaf_reg": 3.0,
    # "rsm": 0.8,            # ❌ QUITAR en GPU
    # "subsample": 0.8,      # (ya lo quitaste por Opción B)
    "bagging_temperature": 1.0,  # estocasticidad compatible con GPU (Bayesian)
    "random_strength": 1.0,
    "random_seed": 42,
    "use_best_model": True,
    "od_type": "Iter",
    "od_wait": 200,
    "task_type": "GPU",
    "verbose": 200,
}

### 4.3. Función conjunta

In [29]:
def run_catboost_experiment(
    metrics_flag: bool,
    model_key: str,
    X_train_scaled,
    y_train,
    X_valid_scaled,
    y_valid,
    resample_rate: float,                 # 0.3, 0.5, 1.0
    catboost_params: dict,
    catboost_metrics_df: pd.DataFrame | None = None,
    n_samples_train: int | None = None,
    n_samples_valid: int | None = None,
    *,
    # extras específicos de CatBoost (opcionales)
    cat_features=None,                    # índices o nombres de columnas categóricas
    use_gpu: bool = False,
    iterations: int = 5000,
    od_wait: int = 200,
    verbose_every: int = 200,
):
    """
    Ejecuta un experimento CatBoost con (opcional) subsampleo de train/valid.

    Parámetros
    ----------
    metrics_flag : bool
        Si True, no entrena y busca métricas previas en catboost_metrics_df[model_key].
    model_key : str
        Nombre/índice del modelo en la tabla de métricas (ej: 'CAT_60_subsampleado_50%').
    X_train_scaled, y_train : arrays
        Ventanas y target de entrenamiento (si ya las tenés escaladas, se usan así).
    X_valid_scaled, y_valid : arrays
        Ventanas y target de validación.
    resample_rate : float
        Proporción a muestrear (0 < r <= 1). 1.0 = sin subsampleo.
    catboost_params : dict
        Hiperparámetros nativos de CatBoost (p. ej., depth, learning_rate, l2_leaf_reg, rsm, subsample, etc.).
    catboost_metrics_df : pd.DataFrame | None
        DataFrame de métricas para leer/escribir (index por model_key). Opcional.
    n_samples_train, n_samples_valid : int | None
        Tamaños base para calcular la cantidad a muestrear. Si es None, se infiere de X_*.

    Retorna
    -------
    dict
        Diccionario con métricas {"RMSE","MAE","R2","SMAPE","DirAcc"}.
    """
    # Si ya hay métricas guardadas y metrics_flag=True, sólo mostrarlas y devolverlas
    if metrics_flag is True and catboost_metrics_df is not None and model_key in catboost_metrics_df.index:
        print("El modelo fue entrenado anteriormente y las métricas ya fueron calculadas\n")
        metrics_dict = catboost_metrics_df.loc[model_key].to_dict()
        print_metrics(metrics_dict, model_key)
        return metrics_dict

    # Mensaje de entrenamiento
    tag_rate = f"{int(resample_rate*100)}%" if resample_rate < 1.0 else "100%"
    print(f"Entrenando modelo {model_key} (resample={tag_rate})...")

    # Calcular tamaños a muestrear (si no los pasan, inferir de X)
    if n_samples_train is None:
        n_samples_train = X_train_scaled.shape[0]
    if n_samples_valid is None:
        n_samples_valid = X_valid_scaled.shape[0]

    # Subsampleo (si corresponde)
    if resample_rate < 1.0:
        n_train_sub = max(1, int(n_samples_train * resample_rate))
        n_valid_sub = max(1, int(n_samples_valid * resample_rate))
        X_train_sub, y_train_sub = subsample(X_train_scaled, y_train, n_train_sub)
        X_valid_sub, y_valid_sub = subsample(X_valid_scaled, y_valid, n_valid_sub)
    else:
        X_train_sub, y_train_sub = X_train_scaled, y_train
        X_valid_sub, y_valid_sub = X_valid_scaled, y_valid

    # Entrenamiento y predicción en valid (usa la función genérica creada para CatBoost)
    model, y_pred_valid = train_model_catboost(
        best_params=catboost_params,
        X_train=X_train_sub, y_train=y_train_sub,
        X_valid=X_valid_sub, y_valid=y_valid_sub,
        cat_features=cat_features,
        use_gpu=use_gpu,
        iterations=iterations,
        od_wait=od_wait,
        verbose_every=verbose_every
    )

    # Evaluación
    metrics_dict = evaluate_model(model, X_valid_sub, y_valid_sub, y_pred=y_pred_valid)

    # Mostrar
    print_metrics(metrics_dict, model_key)

    # Persistir métricas en el DataFrame si se pasa
    if catboost_metrics_df is not None:
        catboost_metrics_df.loc[model_key] = metrics_dict

    return metrics_dict

### 4.4. Función para subsamplear

In [30]:
def subsample(X, y, n):
    n = min(n, X.shape[0])
    idx = np.random.choice(X.shape[0], size=n, replace=False)
    return X[idx], y[idx]

In [41]:
#xgb_device_test = xgb.XGBRegressor(tree_method="gpu_hist", predictor="gpu_predictor")
#try:
#    xgb_device_test.fit([[0,0],[1,1]], [0,1])
#    print("✅ XGBoost GPU works correctly")
#except Exception as e:
#    print("❌ GPU not available for XGBoost:", e)

## 5. Entrenamiento

### 5.1. Entrenamiento 30min

#### 5.1.1. Con 30% de dataset

In [ ]:
cat_30_30 = run_catboost_experiment(
    metrics_flag=False,
    model_key="CAT_30_subsampleado_30%",
    X_train_scaled=X_train_30_scaled, y_train=y_train_30,
    X_valid_scaled=X_valid_30_scaled, y_valid=y_valid_30,
    resample_rate=0.3,
    catboost_params=catboost_default_params,   # el diccionario baseline que te pasé
    catboost_metrics_df=cat_metrics,        # DataFrame opcional para persistir
    use_gpu=True
)

#Tiempo:

Entrenando modelo CAT_30_subsampleado_30% (resample=30%)...
0:	learn: 0.0027632	test: 0.0033834	best: 0.0033834 (0)	total: 72.2ms	remaining: 6m
200:	learn: 0.0021730	test: 0.0030053	best: 0.0030053 (200)	total: 6.21s	remaining: 2m 28s
400:	learn: 0.0020389	test: 0.0029760	best: 0.0029759 (399)	total: 13s	remaining: 2m 29s
600:	learn: 0.0019491	test: 0.0029614	best: 0.0029614 (600)	total: 18.8s	remaining: 2m 17s
800:	learn: 0.0018744	test: 0.0029504	best: 0.0029504 (800)	total: 25.2s	remaining: 2m 12s
1000:	learn: 0.0018105	test: 0.0029411	best: 0.0029411 (1000)	total: 31.7s	remaining: 2m 6s
1200:	learn: 0.0017532	test: 0.0029364	best: 0.0029364 (1173)	total: 38s	remaining: 2m
1400:	learn: 0.0017010	test: 0.0029323	best: 0.0029323 (1400)	total: 44.9s	remaining: 1m 55s
1600:	learn: 0.0016527	test: 0.0029275	best: 0.0029274 (1598)	total: 51s	remaining: 1m 48s
1800:	learn: 0.0016081	test: 0.0029251	best: 0.0029251 (1800)	total: 58.1s	remaining: 1m 43s


#### 5.1.2. Con 50% de dataset

In [ ]:
cat_30_50 = run_catboost_experiment(
    metrics_flag=metrics,
    model_key="CAT_30_subsampleado_50%",
    X_train_scaled=X_train_30_scaled, y_train=y_train_30,
    X_valid_scaled=X_valid_30_scaled, y_valid=y_valid_30,
    resample_rate=0.5,
    catboost_params=catboost_default_params,   # el diccionario baseline que te pasé
    catboost_metrics_df=cat_metrics,        # DataFrame opcional para persistir
    use_gpu=True
)
#Tiempo ~6min

Entrenando modelo LGBM_30_subsampleado_50% (resample=50%)...
Training until validation scores don't improve for 200 rounds
[200]	valid_0's rmse: 0.00274127
[400]	valid_0's rmse: 0.00273439
[600]	valid_0's rmse: 0.00273253
[800]	valid_0's rmse: 0.00273078
[1000]	valid_0's rmse: 0.00272962
[1200]	valid_0's rmse: 0.00272724
[1400]	valid_0's rmse: 0.00272687
[1600]	valid_0's rmse: 0.00272693
[1800]	valid_0's rmse: 0.00272706
Early stopping, best iteration is:
[1631]	valid_0's rmse: 0.00272646
Métricas de LGBM_30_subsampleado_50%:

	 RMSE:	 0.002726
	  MAE:	 0.001601
	   R2:	 0.287471
	SMAPE:	 114.231020
	DirAcc:	 0.711495


#### 5.1.3. Con ventanas completas

In [ ]:
#lgbm_30_100 = run_lgbm_experiment(
#    metrics_flag=False,
#    model_key="LGBM_30_100%",
#    X_train_scaled=X_train_30_scaled, y_train=y_train_30,
#    X_valid_scaled=X_valid_30_scaled, y_valid=y_valid_30,
#    resample_rate=1.0,
#    lgbm_params=lgbm_default_params,
#    lgbm_metrics_df=lgbm_metrics,
#    n_samples_train=n_samples_train_30,
#    n_samples_valid=n_samples_valid_30
#)

Entrenando modelo XGB_30_100% (resample=100%)...
Métricas de XGB_30_100%:

	 RMSE:	 0.002774
	  MAE:	 0.001606
	   R2:	 0.281533
	SMAPE:	 114.170563
	DirAcc:	 0.711309


### 4.2. Entrenamiento 60min

#### 4.2.1. Con 30% de dataset

In [ ]:
lgbm_60_30 = run_lgbm_experiment(
    metrics_flag=metrics,
    model_key="LGBM_60_subsampleado_30%",
    X_train_scaled=X_train_60_scaled, y_train=y_train_60,
    X_valid_scaled=X_valid_60_scaled, y_valid=y_valid_60,
    resample_rate=0.3,
    lgbm_params=lgbm_default_params,
    lgbm_metrics_df=lgbm_metrics,
    n_samples_train=n_samples_train_60,
    n_samples_valid=n_samples_valid_60
)
#Tiempo 60_30: 13.5min (805.149s)

Entrenando modelo LGBM_60_subsampleado_30% (resample=30%)...
Training until validation scores don't improve for 200 rounds
[200]	valid_0's rmse: 0.00357952
[400]	valid_0's rmse: 0.00354671
[600]	valid_0's rmse: 0.00353707
[800]	valid_0's rmse: 0.00352859
[1000]	valid_0's rmse: 0.00352091
[1200]	valid_0's rmse: 0.00351354
[1400]	valid_0's rmse: 0.00351133
[1600]	valid_0's rmse: 0.00350606
[1800]	valid_0's rmse: 0.0035038
[2000]	valid_0's rmse: 0.00350244
[2200]	valid_0's rmse: 0.00350058
[2400]	valid_0's rmse: 0.00349925
[2600]	valid_0's rmse: 0.00349649
[2800]	valid_0's rmse: 0.00349481
[3000]	valid_0's rmse: 0.00349447
[3200]	valid_0's rmse: 0.0034942
[3400]	valid_0's rmse: 0.00349313
[3600]	valid_0's rmse: 0.00349281
Early stopping, best iteration is:
[3568]	valid_0's rmse: 0.0034925
Métricas de LGBM_60_subsampleado_30%:

	 RMSE:	 0.003493
	  MAE:	 0.001892
	   R2:	 0.395036
	SMAPE:	 95.774103
	DirAcc:	 0.781957


#### 4.2.2. Con 50% de dataset

In [ ]:
lgbm_60_50 = run_lgbm_experiment(
    metrics_flag=metrics,
    model_key="LGBM_60_subsampleado_50%",
    X_train_scaled=X_train_60_scaled, y_train=y_train_60,
    X_valid_scaled=X_valid_60_scaled, y_valid=y_valid_60,
    resample_rate=0.5,
    lgbm_params=lgbm_default_params,
    lgbm_metrics_df=lgbm_metrics,
    n_samples_train=n_samples_train_60,
    n_samples_valid=n_samples_valid_60
)
#Tiempo 60_50:    18.73min ( 1123,93 s)

Entrenando modelo LGBM_60_subsampleado_50% (resample=50%)...
Training until validation scores don't improve for 200 rounds
[200]	valid_0's rmse: 0.00373237
[400]	valid_0's rmse: 0.00368379
[600]	valid_0's rmse: 0.00366461
[800]	valid_0's rmse: 0.00364943
[1000]	valid_0's rmse: 0.00363785
[1200]	valid_0's rmse: 0.00363024
[1400]	valid_0's rmse: 0.00362678
[1600]	valid_0's rmse: 0.00362023
[1800]	valid_0's rmse: 0.00361772
[2000]	valid_0's rmse: 0.00361273
[2200]	valid_0's rmse: 0.00361155
[2400]	valid_0's rmse: 0.00360977
[2600]	valid_0's rmse: 0.00360707
[2800]	valid_0's rmse: 0.00360605
[3000]	valid_0's rmse: 0.00360455
[3200]	valid_0's rmse: 0.0036032
[3400]	valid_0's rmse: 0.003602
[3600]	valid_0's rmse: 0.00360134
[3800]	valid_0's rmse: 0.00360106
[4000]	valid_0's rmse: 0.00360045
[4200]	valid_0's rmse: 0.0035997
[4400]	valid_0's rmse: 0.00359902
[4600]	valid_0's rmse: 0.00359873
[4800]	valid_0's rmse: 0.00359794
[5000]	valid_0's rmse: 0.00359789
Did not meet early stopping. Best i

#### 4.2.3. Con ventanas completas

In [ ]:
#lgbm_60_30 = run_lgbm_experiment(
#    metrics_flag=False,
#    model_key="LGBM_60_100%",
#    X_train_scaled=X_train_60_scaled, y_train=y_train_60,
#    X_valid_scaled=X_valid_60_scaled, y_valid=y_valid_60,
#    resample_rate=1,
#    lgbm_params=lgbm_default_params,
#    lgbm_metrics_df=lgbm_metrics,
#    n_samples_train=n_samples_train_60,
#    n_samples_valid=n_samples_valid_60
#)


Entrenando modelo XGB_60_100% (resample=100%)...
Métricas de XGB_60_100%:

	 RMSE:	 0.003547
	  MAE:	 0.001919
	   R2:	 0.416467
	SMAPE:	 95.899622
	DirAcc:	 0.786994


### 4.3. Entrenamiento 90min

#### 4.3.1. Con 30% de dataset

In [ ]:
lgbm_90_30 = run_lgbm_experiment(
    metrics_flag=metrics,
    model_key="LGBM_90_subsampleado_30%",
    X_train_scaled=X_train_90_scaled, y_train=y_train_90,
    X_valid_scaled=X_valid_90_scaled, y_valid=y_valid_90,
    resample_rate=0.3,
    lgbm_params=lgbm_default_params,
    lgbm_metrics_df=lgbm_metrics,
    n_samples_train=n_samples_train_90,
    n_samples_valid=n_samples_valid_90
)
#Tiempo 90_30: 13.7min (763.253s)

Entrenando modelo LGBM_30_subsampleado_30% (resample=30%)...
Training until validation scores don't improve for 200 rounds
[200]	valid_0's rmse: 0.00459369
[400]	valid_0's rmse: 0.00449777
[600]	valid_0's rmse: 0.00446398
[800]	valid_0's rmse: 0.00443929
[1000]	valid_0's rmse: 0.00442442
[1200]	valid_0's rmse: 0.00441426
[1400]	valid_0's rmse: 0.00440755
[1600]	valid_0's rmse: 0.00440237
[1800]	valid_0's rmse: 0.00439761
[2000]	valid_0's rmse: 0.00439349
[2200]	valid_0's rmse: 0.0043916
[2400]	valid_0's rmse: 0.00439026
[2600]	valid_0's rmse: 0.00438794
[2800]	valid_0's rmse: 0.00438671
[3000]	valid_0's rmse: 0.00438549
[3200]	valid_0's rmse: 0.00438423
[3400]	valid_0's rmse: 0.004384
Early stopping, best iteration is:
[3343]	valid_0's rmse: 0.0043834
Métricas de LGBM_30_subsampleado_30%:

	 RMSE:	 0.004383
	  MAE:	 0.002170
	   R2:	 0.450930
	SMAPE:	 87.363224
	DirAcc:	 0.812991


In [ ]:
lgbm_metrics

,RMSE,MAE,R2,SMAPE,DirAcc
LGBM_90_subsampleado_30%,0.004383,0.002170,0.450930,87.363224,0.812991
LGBM_30_subsampleado_50%,0.002726,0.001601,0.287471,114.231020,0.711495
LGBM_60_subsampleado_30%,0.003493,0.001892,0.395036,95.774103,0.781957
LGBM_30_subsampleado_30%,0.002875,0.001621,0.268676,114.153659,0.709463
LGBM_60_subsampleado_50%,0.003598,0.001905,0.405742,96.080063,0.785594


#### 4.3.2. Con 50% de dataset

In [ ]:
lgbm_90_50 = run_lgbm_experiment(
    metrics_flag=metrics,
    model_key="LGBM_90_subsampleado_50%",
    X_train_scaled=X_train_90_scaled, y_train=y_train_90,
    X_valid_scaled=X_valid_90_scaled, y_valid=y_valid_90,
    resample_rate=0.5,
    lgbm_params=lgbm_default_params,
    lgbm_metrics_df=lgbm_metrics,
    n_samples_train=n_samples_train_90,
    n_samples_valid=n_samples_valid_90
)

Entrenando modelo LGBM_90_subsampleado_50% (resample=50%)...
Training until validation scores don't improve for 200 rounds
[200]	valid_0's rmse: 0.00470718
[400]	valid_0's rmse: 0.00457675
[600]	valid_0's rmse: 0.00452857
[800]	valid_0's rmse: 0.00449961
[1000]	valid_0's rmse: 0.00448594
[1200]	valid_0's rmse: 0.00447562
[1400]	valid_0's rmse: 0.00447053
[1600]	valid_0's rmse: 0.00446672
[1800]	valid_0's rmse: 0.00446337
[2000]	valid_0's rmse: 0.0044618
[2200]	valid_0's rmse: 0.00445926
[2400]	valid_0's rmse: 0.00445622
[2600]	valid_0's rmse: 0.00445505
[2800]	valid_0's rmse: 0.00445317
[3000]	valid_0's rmse: 0.00445243
Early stopping, best iteration is:
[2920]	valid_0's rmse: 0.00445181
Métricas de LGBM_90_subsampleado_50%:

	 RMSE:	 0.004452
	  MAE:	 0.002165
	   R2:	 0.448462
	SMAPE:	 87.058502
	DirAcc:	 0.817688


#### 4.3.3. Con ventanas completas

In [ ]:
#lgbm_90_100 = run_lgbm_experiment(
#    metrics_flag=False,
#    model_key="LGBM_90_100%",
#    X_train_scaled=X_train_90_scaled, y_train=y_train_90,
#    X_valid_scaled=X_valid_90_scaled, y_valid=y_valid_90,
#    resample_rate=1.0,
#    lgbm_params=lgbm_default_params,
 #   lgbm_metrics_df=lgbm_metrics,
 #   n_samples_train=n_samples_train_90,
 #   n_samples_valid=n_samples_valid_90
#)


Entrenando modelo XGB_90_100% (resample=100%)...
Métricas de XGB_90_100%:

	 RMSE:	 0.004334
	  MAE:	 0.002167
	   R2:	 0.457986
	SMAPE:	 87.189119
	DirAcc:	 0.818053


## 5. Recuperación de métricas

In [ ]:
save_metrics(lgbm_metrics, "4_3_lgbm_metrics")

In [ ]:
lgbm_metrics

,RMSE,MAE,R2,SMAPE,DirAcc
LGBM_90_subsampleado_30%,0.004383,0.002170,0.450930,87.363224,0.812991
LGBM_30_subsampleado_50%,0.002726,0.001601,0.287471,114.231020,0.711495
LGBM_60_subsampleado_30%,0.003493,0.001892,0.395036,95.774103,0.781957
LGBM_30_subsampleado_30%,0.002875,0.001621,0.268676,114.153659,0.709463
LGBM_60_subsampleado_50%,0.003598,0.001905,0.405742,96.080063,0.785594
LGBM_90_subsampleado_50%,0.004452,0.002165,0.448462,87.058502,0.817688


Este punto existe para garantizar reproducibilidad y continuidad del análisis sin reentrenar modelos cuando se pierden las métricas. Actúa como fallback: reconstruye la tabla de métricas de Random Forest a partir de valores ya validados y la persiste nuevamente, evitando el re-entrenamiento de los modelos.

Así, se mantiene la consistencia de resultados y la trazabilidad de comparaciones y conclusiones, incluso si el archivo original fue eliminado, corrompido o el entorno de ejecución cambió.

In [ ]:
def generate_metrics_lgbm():
  '''
  Ejecutar está función solo en caso de perder las métricas
  El objetivo es no volver a correr los entrenamientos
  '''
  lgbm_metrics = pd.DataFrame(columns=["RMSE", "MAE", "R2", "SMAPE", "DirAcc"])

  lgbm_30_30 = {
      "RMSE": 0.002875,
      "MAE": 0.001621,
      "R2": 0.268676,
      "SMAPE": 114.153659,
      "DirAcc": 0.709463
  }

  lgbm_30_50 = {
      "RMSE": 0.002726,
      "MAE": 0.001601,
      "R2": 0.287471,
      "SMAPE": 114.231020,
      "DirAcc": 0.711495
  }

  lgbm_60_30 = {
      "RMSE": 0.003493,
      "MAE": 0.001892,
      "R2": 0.395036,
      "SMAPE": 95.774103,
      "DirAcc": 0.781957
  }

  lgbm_60_50 = {
      "RMSE": 0.003598,
      "MAE": 0.001905,
      "R2": 0.405742,
      "SMAPE": 96.080063,
      "DirAcc": 0.785594
  }

  lgbm_90_30 = {
      "RMSE": 0.004383,
      "MAE": 0.002170,
      "R2": 0.450930,
      "SMAPE": 87.363224,
      "DirAcc": 0.812991
  }


  lgbm_90_50 = {
      "RMSE": 0.004452,
      "MAE":  0.002165,
      "R2": 0.448462,
      "SMAPE": 87.058502,
      "DirAcc": 0.817688
      }

  lgbm_metrics.loc['LGBM_30_subsampleado_30%'] = lgbm_30_30
  lgbm_metrics.loc['LGBM_30_subsampleado_50%'] = lgbm_30_50

  lgbm_metrics.loc['LGBM_60_subsampleado_30%'] = lgbm_60_30
  lgbm_metrics.loc['LGBM_60_subsampleado_50%'] = lgbm_60_50

  lgbm_metrics.loc['LGBM_90_subsampleado_30%'] = lgbm_90_30
  lgbm_metrics.loc['LGBM_90_subsampleado_50%'] = lgbm_90_50


  save_metrics(lgbm_metrics, "4_3_lgbm_metrics")
  return lgbm_metrics

In [ ]:
#Ejecutar esta función solo en caso de perder las métricas de entrenamiento
#lgbm_metrics = generate_metrics_lgbm()

Métricas guardadas en /content/drive/MyDrive/neural_profit/4_model_training/4_3_lgbm_metrics.parquet


In [ ]:
lgbm_metrics

,RMSE,MAE,R2,SMAPE,DirAcc
LGBM_30_subsampleado_30%,0.002875,0.001621,0.268676,114.153659,0.709463
LGBM_30_subsampleado_50%,0.002726,0.001601,0.287471,114.231020,0.711495
LGBM_60_subsampleado_30%,0.003493,0.001892,0.395036,95.774103,0.781957
LGBM_60_subsampleado_50%,0.003598,0.001905,0.405742,96.080063,0.785594
LGBM_90_subsampleado_30%,0.004383,0.002170,0.450930,87.363224,0.812991
LGBM_90_subsampleado_50%,0.004452,0.002165,0.448462,87.058502,0.817688


Métricas guardadas en /content/drive/MyDrive/neural_profit/4_model_training/4_3_lgbm_metrics.parquet
